# Steering Alert Threshold Comparison

Evaluates four threshold methods on `field_session_042`.

**Production decision:** `Mean + 2σ` (see `src/analytics.py` → `calculate_steering_threshold`).

Note: `quality.py` still uses IQR for **sensor outlier masking** — that is separate from steering alert thresholds.

In [ ]:
import sys
from pathlib import Path

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd

from ingestion import (
    MeasurementColumn,
    load_dataset,
    load_session_metadata,
    normalize_types,
    prepare_measurement_data,
    add_session_timeline,
    validate_required_columns,
    validate_metadata_file,
)
from quality import build_quality_dataframe, build_analytics_dataframe
from analytics import calculate_steering_threshold, build_analytics_bundle

CSV = ROOT / "sample-data" / "field_session_042.csv"
META = ROOT / "sample-data" / "metadata_session_042.json"

raw = load_dataset(CSV)
meta = load_session_metadata(META)
validate_required_columns(raw)
validate_metadata_file(meta)

df = add_session_timeline(prepare_measurement_data(normalize_types(raw)), meta)
qdf = build_quality_dataframe(df)
adf = build_analytics_dataframe(df, qdf)

wheel_delta = adf[MeasurementColumn.WHEEL_ANGLE].diff().abs()
same_ctx = adf[MeasurementColumn.REVERSE_STATE] == adf[MeasurementColumn.REVERSE_STATE].shift(1)

In [ ]:
def context_deltas(reverse_state):
    mask = (adf[MeasurementColumn.REVERSE_STATE] == reverse_state) & same_ctx
    return wheel_delta.loc[mask].dropna()


def threshold_fixed_5(deltas):
    return 5.0


def threshold_mean_2std(deltas):
    return calculate_steering_threshold(deltas)


def threshold_q3_iqr(deltas):
    q1, q3 = deltas.quantile(0.25), deltas.quantile(0.75)
    iqr = q3 - q1
    return q3 + 1.5 * iqr if iqr > 0 else deltas.mean() + 2 * deltas.std()


def threshold_p95(deltas):
    return deltas.quantile(0.95)


def find_alerts(reverse_state, threshold):
    mask = (
        (adf[MeasurementColumn.REVERSE_STATE] == reverse_state)
        & same_ctx
        & (wheel_delta > threshold)
    )
    hits = adf.loc[mask, [MeasurementColumn.ELAPSED_SECONDS]].copy()
    hits["wheel_delta"] = wheel_delta[mask]
    return hits


METHODS = {
    "Fixed 5°": threshold_fixed_5,
    "Mean + 2σ (production)": threshold_mean_2std,
    "Q3 + 1.5×IQR (historical)": threshold_q3_iqr,
    "95th percentile": threshold_p95,
}

In [ ]:
rows = []
for context, rs in [("Forward", 0), ("Reverse", 1)]:
    deltas = context_deltas(rs)
    for label, fn in METHODS.items():
        thr = fn(deltas)
        alerts = find_alerts(rs, thr)
        rows.append({
            "context": context,
            "method": label,
            "n_deltas": len(deltas),
            "threshold_deg": round(thr, 3),
            "alert_count": len(alerts),
            "alert_seconds": alerts[MeasurementColumn.ELAPSED_SECONDS].tolist(),
            "alert_deltas": alerts["wheel_delta"].round(2).tolist(),
        })

pd.DataFrame(rows)

In [ ]:
bundle = build_analytics_bundle(adf, sample_rate_hz=meta["sample_rate_hz"])
print("Production bundle (Mean + 2σ):")
print("  Forward threshold:", round(bundle["forward_metrics"]["steering_threshold"], 3))
print("  Reverse threshold:", round(bundle["reverse_metrics"]["steering_threshold"], 3))
print("  Forward alerts:", bundle["forward_metrics"]["sudden_steering_events"])
print("  Reverse alerts:", bundle["reverse_metrics"]["sudden_steering_events"])

## Recommendation

**Mean + 2σ** was chosen for production because it is easy to explain, adapts per context, and produces meaningful alerts on the sample session without a fixed arbitrary cutoff.

IQR-based steering thresholds are retained in this notebook for historical comparison only.